# Internal-scope phase check — is the 42° from the FPGA or the measuring rig?

Uses the Red Pitaya's **built-in acquisition (scope) block** (still present in your bitstream on
`sys[1]`, base `0x40100000`) to measure the FPGA input and output *inside the board*, with no
external scope, no 50/50 divider, and no BNC runs in the comparison.

## Wiring
Add ONE short cable: **OUT1 → IN2** (loopback). Keep the signal generator on **IN1** as usual.
- CH A of the internal scope = **IN1** = FPGA input
- CH B of the internal scope = **IN2** = FPGA output (via DAC → cable → ADC)

Both channels are digitized by identical ADC chains at the same clock and stored at the same
buffer index, so their **relative** phase is exact and common latency cancels.

## How to read the result
1. Set the FPGA to phase 0 and run a measurement. The reported CHB−CHA phase is the FPGA+DAC+ADC
   path only (no external rig).
2. Run the sweep: it steps the FPGA setpoint 0→180° and checks the measured phase moves 1:1.
   - Slope ≈ 1 and small constant offset  → the module applies phase faithfully; the offset is
     fixed board transport delay. Whatever the external scope shows *beyond* this offset is the
     divider/cable artifact.
   - Measured ≈ external 42° here too      → the lag is real board latency, not the rig.

*Python 3.5 compatible (no f-strings). Runs as root.*

In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6

# ---------------- phase-shifter (sys[3]) : reused to set FPGA phase -------------
PS_BASE=0x40300000; SCALE=1<<14; L_PIPE=2
class PhaseShifter(object):
    def __init__(self):
        self.fd=os.open('/dev/mem',os.O_RDWR|os.O_SYNC)
        self.m=mmap.mmap(self.fd,0x1000,offset=PS_BASE)
    def _w(self,o,v): self.m[o:o+4]=struct.pack('<I',v&0xFFFFFFFF)
    @staticmethod
    def _s16(x): x=int(round(x)); return max(-32768,min(32767,x))&0xFFFF
    def configure(self,phase_deg,gain,fsig):
        phi=np.deg2rad(phase_deg-360.0*L_PIPE*fsig/FS_CLK)
        D=FS_CLK/(4.0*fsig); dly=min(int(np.floor(D)),1022)
        delta=min(int(round((D-np.floor(D))*SCALE)),SCALE-1)
        self._w(0x00,self._s16(gain*np.cos(phi)*SCALE))
        self._w(0x04,self._s16(gain*np.sin(phi)*SCALE))
        self._w(0x08,dly); self._w(0x0C,delta); self._w(0x10,0)

# ---------------- internal scope (sys[1]) -------------------------------------
SC_BASE=0x40100000; SC_MAP=0x40000
BUF_A=0x10000; BUF_B=0x20000; NMAX=1<<14   # 16384 samples/chan
# Red Pitaya default equalization-filter coeffs (LV / +-1 V jumper). Same on both
# channels, so any residual filter phase is identical and cancels in CHB-CHA.
FILT=dict(AA=0x7D93, BB=0x437C7, KK=0xd9999a, PP=0x2666)

class Scope(object):
    def __init__(self):
        self.fd=os.open('/dev/mem',os.O_RDWR|os.O_SYNC)
        self.m=mmap.mmap(self.fd,SC_MAP,offset=SC_BASE)
        self._set_filters()
    def _w(self,o,v): self.m[o:o+4]=struct.pack('<I',v&0xFFFFFFFF)
    def _r(self,o):   return struct.unpack('<I',self.m[o:o+4])[0]
    def _set_filters(self):
        for ch,base in (('a',0x30),('b',0x40)):
            self._w(base+0x0,FILT['AA']); self._w(base+0x4,FILT['BB'])
            self._w(base+0x8,FILT['KK']); self._w(base+0xC,FILT['PP'])
    def _buf(self, off, n):
        # IMPORTANT: read the buffer ONE 32-bit word at a time.
        # A bulk slice (self.m[off:off+4*n]) becomes a wide/burst memcpy that the
        # GP-AXI -> sys-bus bridge cannot ack -> AXI lockup -> board reboots.
        m = self.m
        ba = bytearray(4*n)
        for i in range(n):
            o = off + 4*i
            ba[4*i:4*i+4] = m[o:o+4]          # single aligned 32-bit read (safe)
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        return np.where(v & 0x2000, v-0x4000, v).astype(float)   # sign-extend 14-bit
    def acquire(self, dec=1, n=NMAX):
        self._w(0x14, dec)        # decimation
        self._w(0x10, n-1)        # samples to record after trigger
        self._w(0x00, 2)          # reset acquisition
        self._w(0x00, 1)          # arm (start writing)
        self._w(0x04, 1)          # software trigger now
        t0=time.time()
        while self._r(0x04)!=0:   # trig-source auto-clears when done
            if time.time()-t0>1.0: break
        time.sleep(0.005)
        fs=FS_CLK/dec
        return self._buf(BUF_A,n), self._buf(BUF_B,n), fs

# ---------------- phase math -------------------------------------------------
def est_freq(x, fs):
    x=x-x.mean(); w=np.hanning(len(x))
    X=np.abs(np.fft.rfft(x*w)); k=int(np.argmax(X[1:]))+1
    if 1<=k<len(X)-1:
        a,b,c=np.log(X[k-1]+1e-9),np.log(X[k]+1e-9),np.log(X[k+1]+1e-9)
        k=k+0.5*(a-c)/(a-2*b+c)
    return k*fs/len(x)

def phases(a, b, f, fs):
    n=np.arange(len(a)); c=np.cos(2*np.pi*f*n/fs); s=np.sin(2*np.pi*f*n/fs)
    def ph(x):
        x=x-x.mean(); return np.dot(x,c)-1j*np.dot(x,s)
    pa,pb=ph(a),ph(b)
    amp_a=2*abs(pa)/len(a); amp_b=2*abs(pb)/len(b)
    dphi=np.degrees(np.angle(pb)-np.angle(pa))
    dphi=(dphi+180)%360-180
    return dphi, amp_a, amp_b

ps=PhaseShifter(); sc=Scope()
print('phase shifter + internal scope mapped OK')

## Single measurement
Set your generator frequency below. This sets the FPGA to phase 0 and measures the true
input→output phase inside the board.

In [ ]:
SIG_FREQ = 740e3      # <-- set to your signal-generator frequency [Hz]

ps.configure(phase_deg=0.0, gain=1.0, fsig=SIG_FREQ)
time.sleep(0.05)
a, b, fs = sc.acquire(dec=1)
f_meas = est_freq(a, fs)
dphi, amp_a, amp_b = phases(a, b, f_meas, fs)

print('measured signal freq : %.1f kHz' % (f_meas/1e3))
print('CHA (input)  amplitude: %.0f counts' % amp_a)
print('CHB (output) amplitude: %.0f counts' % amp_b)
print('phase(output) - phase(input) at FPGA setpoint 0 deg : %+.2f deg' % dphi)
if amp_b < 50:
    print('   ** CHB is nearly flat -- check the OUT1 -> IN2 loopback cable **')

### Plot a few periods of both channels

In [ ]:
import matplotlib
matplotlib.use('Agg') if False else None
import matplotlib.pyplot as plt

per = fs/f_meas
win = int(min(len(a), round(4*per)))          # ~4 periods
t = np.arange(win)/fs*1e6                       # microseconds
plt.figure(figsize=(9,4))
plt.plot(t, a[:win], label='CH A  IN1 (FPGA input)')
plt.plot(t, b[:win], label='CH B  IN2 (FPGA output, looped)')
plt.xlabel('time [us]'); plt.ylabel('ADC counts')
plt.title('Internal scope: input vs output  (dphi = %+.2f deg)' % dphi)
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## Phase sweep — the decisive test
Steps the FPGA setpoint and measures the internal phase. A straight line of **slope ≈ 1**
means the module applies exactly the phase you request; the **intercept** is the fixed board
transport delay. Any extra phase your *external* scope shows beyond this intercept is the
divider/cable artifact.

In [ ]:
sets = np.arange(0, 181, 30)      # FPGA setpoints [deg]
meas = []
for phi in sets:
    ps.configure(phase_deg=float(phi), gain=1.0, fsig=SIG_FREQ)
    time.sleep(0.05)
    a,b,fs = sc.acquire(dec=1)
    fm = est_freq(a, fs)
    d,_,_ = phases(a,b,fm,fs)
    meas.append(d)
    print('setpoint %+6.1f deg  ->  measured %+7.2f deg' % (phi, d))
meas=np.unwrap(np.radians(meas)); meas=np.degrees(meas)

# linear fit measured = slope*setpoint + offset
# (np.polyfit is stable across numpy versions; old numpy's lstsq rejects rcond=None)
slope, offset = np.polyfit(sets.astype(float), np.asarray(meas, dtype=float), 1)
print('\nlinear fit:  measured = %.3f * setpoint + (%.2f deg)' % (slope, offset))
print('slope ~ -1 expected (FPGA lags by setpoint). Constant offset = board transport delay.')

plt.figure(figsize=(6,5))
plt.plot(sets, meas, 'o-', label='measured')
plt.plot(sets, slope*sets+offset, '--', label='fit')
plt.xlabel('FPGA phase setpoint [deg]'); plt.ylabel('internal CHB-CHA phase [deg]')
plt.title('Does the FPGA apply what you set?'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()